In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
 
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"
]
 
drop_cols = [
    "difficulty",
    "srv_serror_rate", "dst_host_srv_serror_rate",
    "srv_rerror_rate", "dst_host_srv_rerror_rate",
]
categorical_cols = ["protocol_type", "service", "flag"]

In [3]:
train_df = pd.read_csv("../data/KDDTrain.txt", names=columns)
test_df  = pd.read_csv("../data/KDDTest.txt",  names=columns)
 
test_labels = test_df["label"].copy()
 
train_df["binary_label"] = train_df["label"].apply(lambda x: 0 if x == "normal" else 1)
test_df["binary_label"]  = test_df["label"].apply(lambda x: 0 if x == "normal" else 1)
 
train_df = train_df.drop(columns=drop_cols + ["label"])
test_df  = test_df.drop(columns=drop_cols  + ["label"])
 
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])
 
X_train = train_df.drop(columns=["binary_label"])
y_train = train_df["binary_label"]
X_test  = test_df.drop(columns=["binary_label"])
y_test  = test_df["binary_label"]
 
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
 
model = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    scale_pos_weight=spw,
    random_state=42, eval_metric="logloss", n_jobs=-1
)
model.fit(X_train, y_train)
 
y_pred = model.predict(X_test)
 
explainer = shap.TreeExplainer(model)
base_val  = explainer.expected_value
 
analysis = pd.DataFrame({
    "true_binary": y_test.values,
    "predicted":   y_pred,
    "true_label":  test_labels.values,
    "missed":      ((y_test.values == 1) & (y_pred == 0)).astype(int),
}, index=X_test.index)

In [4]:
def waterfall_plot(row_index, label, save_path):
    row      = X_test.loc[[row_index]]
    sv       = explainer.shap_values(row)[0]
    true_lbl = analysis.loc[row_index, "true_label"]
    pred     = analysis.loc[row_index, "predicted"]
 
    explanation = shap.Explanation(
        values=sv,
        base_values=base_val,
        data=row.iloc[0].values,
        feature_names=list(X_test.columns)
    )
 
    plt.figure(figsize=(10, 7))
    shap.plots.waterfall(explanation, show=False, max_display=15)
    plt.title(
        f"{label} | True: {true_lbl} | Predicted: {'ATTACK' if pred==1 else 'NORMAL'}",
        fontsize=10
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}  (true={true_lbl}, pred={'attack' if pred==1 else 'normal'})")

In [5]:
caught_idx = analysis[(analysis["true_binary"]==1) & (analysis["predicted"]==1)].index[0]
missed_idx = analysis[(analysis["true_binary"]==1) & (analysis["predicted"]==0)].index[0]
normal_idx = analysis[(analysis["true_binary"]==0) & (analysis["predicted"]==0)].index[0]
gp_idx     = analysis[analysis["true_label"]=="guess_passwd"].index[0]
 
print("--- Waterfall plots ---")
waterfall_plot(caught_idx, "Caught Attack",         "wf_caught_attack.png")
waterfall_plot(missed_idx, "Missed Attack",         "wf_missed_attack.png")
waterfall_plot(normal_idx, "Correct Normal",        "wf_correct_normal.png")
waterfall_plot(gp_idx,     "guess_passwd (Missed)", "wf_guess_passwd.png")

--- Waterfall plots ---
Saved: wf_caught_attack.png  (true=neptune, pred=attack)
Saved: wf_missed_attack.png  (true=mscan, pred=normal)
Saved: wf_correct_normal.png  (true=normal, pred=normal)
Saved: wf_guess_passwd.png  (true=guess_passwd, pred=normal)


In [6]:
def cosine_sim(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return np.dot(a, b) / denom

normal_rows   = analysis[analysis["true_binary"] == 0].index
normal_sample = X_test.loc[normal_rows].sample(min(500, len(normal_rows)), random_state=42)
normal_shap   = explainer.shap_values(normal_sample)
mean_shap_normal = normal_shap.mean(axis=0)

caught_rows   = analysis[(analysis["true_binary"]==1) & (analysis["predicted"]==1)].index
caught_sample = X_test.loc[caught_rows].sample(min(500, len(caught_rows)), random_state=42)
caught_shap   = explainer.shap_values(caught_sample)
mean_shap_caught = caught_shap.mean(axis=0)

attack_types = analysis[analysis["true_binary"]==1]["true_label"].unique()
 
results = []
for attack in sorted(attack_types):
    rows   = analysis[analysis["true_label"] == attack].index
    sample = X_test.loc[rows].sample(min(100, len(rows)), random_state=42)
    sv     = explainer.shap_values(sample)
    mean_sv = sv.mean(axis=0)
 
    sim_to_normal = cosine_sim(mean_sv, mean_shap_normal)
    sim_to_caught = cosine_sim(mean_sv, mean_shap_caught)
    total         = len(rows)
    missed        = analysis.loc[rows, "missed"].sum()
    recall        = 1 - (missed / total)
    camouflage    = sim_to_normal - sim_to_caught
 
    results.append({
        "attack_type":      attack,
        "total":            total,
        "recall":           round(recall, 3),
        "sim_to_normal":    round(sim_to_normal, 3),
        "sim_to_caught":    round(sim_to_caught, 3),
        "camouflage_score": round(camouflage, 3),
    })
 
flag_df = pd.DataFrame(results).sort_values("camouflage_score", ascending=False)
 
print("\n=== ATTACK TYPE FLAGGING - sorted by camouflage score ===")
print(flag_df.to_string(index=False))
 
flagged = flag_df[flag_df["camouflage_score"] > 0]
print(f"\n=== FLAGGED as SHAP-camouflaged ({len(flagged)} attack types) ===")
print(flagged[["attack_type","total","recall","camouflage_score"]].to_string(index=False))


=== ATTACK TYPE FLAGGING - sorted by camouflage score ===
    attack_type  total  recall  sim_to_normal  sim_to_caught  camouflage_score
  snmpgetattack    178   0.000          0.949         -0.822             1.772
   guess_passwd   1231   0.000          0.895         -0.738             1.633
       mailbomb    293   0.000          0.820         -0.784             1.605
        rootkit     13   0.000          0.612         -0.905             1.517
           worm      2   0.000          0.608         -0.853             1.460
          xlock      9   0.000          0.582         -0.778             1.361
      snmpguess    331   0.000          0.785         -0.560             1.345
          xterm     13   0.231          0.526         -0.799             1.326
         xsnoop      4   0.250          0.505         -0.791             1.296
       multihop     18   0.278          0.431         -0.815             1.246
           imap      1   0.000          0.467         -0.764            

In [7]:
plt.figure(figsize=(10, 7))
for _, row in flag_df.iterrows():
    size  = max(30, row["total"] / 5)
    color = "tomato" if row["camouflage_score"] > 0 else "steelblue"
    plt.scatter(row["camouflage_score"], row["recall"], s=size, color=color, alpha=0.7)
    plt.annotate(
        row["attack_type"],
        (row["camouflage_score"], row["recall"]),
        textcoords="offset points", xytext=(5, 3), fontsize=7
    )
 
plt.axvline(x=0, color="black", linestyle="--", alpha=0.4)
plt.xlabel("Camouflage Score (sim to normal − sim to caught attacks)")
plt.ylabel("Recall")
plt.title("Camouflage Score vs Recall per Attack Type\n"
          "(red = flagged, dot size = instance count)")
from matplotlib.patches import Patch
plt.legend(handles=[
    Patch(color="tomato",    label="Flagged (camouflaged)"),
    Patch(color="steelblue", label="Not flagged"),
], loc="lower left")
plt.tight_layout()